# Práctica 3: Soluciones de los ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [NLTK](https://www.nltk.org) de Python.

### Ejercicio 1

El objetivo de este ejercicio es entrenar y evaluar el rendimiento de un filtro de correo electrónico no deseado. Para ello se usará el corpus Enron-Spam, pero no se proporcionará un vocabulario fijo, sino que este deberá aprenderse a partir de los mensajes de entrenamiento. Con el objetivo de homogeneizar el vocabulario aprendido y de mejorar el rendimiento del filtro construido, se pedirá que se apliquen distintas técnicas de preprocesado.

En todos los apartados de este ejercicio se deberá realizar lo siguiente:

* Construir el filtro como una tubería de scikit-learn que concatene un vectorizador tf-idf y un modelo $k$NN clasificador con 5 vecinos y que use la métrica del coseno.
* Definir una función `procesa_mensaje` que, dado el contenido en bruto de un mensaje, aplique todos los pasos de procesamiento pedidos hasta obtener la lista de tókenes correspondiente. Esta función se deberá proporcionar como argumento `analyzer` del vectorizador tf-idf.
* Entrenar el filtro con el corpus de entrenamiento.
* Calcular la sensibilidad del filtro sobre el corpus de prueba.

In [ ]:
from email import parser
from email import policy

In [ ]:
analizador_mensaje = parser.Parser(policy=policy.default)

In [ ]:
from pathlib import Path

In [ ]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

#### Apartado 0

En este apartado se pide procesar los mensajes realizando los siguientes 3 pasos:

* Extraer el contenido de texto de los mensajes en formato HTML. Para ello hacer uso de la biblioteca [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).
* Dividir el contenido de los mensajes en secuencias de tókenes mediante el tokenizador de NLTK.
* Eliminar de los tókenes los caracteres no alfanuméricos (y eliminar por completo aquellos tókenes que no contengan caracteres alfanuméricos).

In [ ]:
contenidos_mensajes_entrenamiento[-21]

**Nota de clase**:
- esta librería sirve para extraer el texto de los mensajes, eliminando las etiquetas HTML y quedándonos solo con el contenido textual en páginas web o correos electrónicos que estén en formato HTML.
- La calidad de los datos influye mucho en la calidad del modelo, por lo que es importante eliminar los caracteres no alfanuméricos para reducir el ruido en los datos y mejorar el rendimiento del filtro de correo electrónico no deseado.

In [20]:
from bs4 import BeautifulSoup

In [21]:
def elimina_html(contenido):
    return BeautifulSoup(contenido).get_text()

In [22]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [25]:
import os

os.environ['NLTK_DATA'] = '.'

from nltk import download

download('punkt', download_dir='.')

download('punkt_tab', download_dir='.')

[nltk_data] Downloading package punkt to ....
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [26]:
from nltk.tokenize import word_tokenize

In [27]:
from pprint import pprint

In [28]:
elimina_html(contenidos_mensajes_entrenamiento[-21])

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [29]:
word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21]))

['``',
 'I',
 'just',
 'wanted',
 'to',
 'write',
 'and',
 'thank',
 'you',
 'for',
 'Spur-M',
 '.',
 'I',
 'suffered',
 'from',
 'poor',
 'sperm',
 'count',
 'and',
 'motility',
 '.',
 'I',
 'found',
 'your',
 'site',
 'and',
 'ordered',
 'Spur-M',
 'Fertility',
 'Blend',
 'for',
 'Men',
 '.',
 'I',
 'have',
 'wondered',
 'for',
 'years',
 'what',
 'caused',
 'low',
 'semen',
 'and',
 'sperm',
 'count',
 ',',
 'and',
 'how',
 'I',
 'could',
 'improve',
 'my',
 'fertility',
 'and',
 'help',
 'my',
 'wife',
 'conceive',
 '.',
 'Spur-M',
 'seems',
 'to',
 'have',
 'done',
 'just',
 'that',
 '!',
 'Thank',
 'you',
 'for',
 'your',
 'support',
 '.',
 "''",
 'Andrew',
 'H.',
 ',',
 'London',
 ',',
 'UK',
 "''",
 'Spur-M',
 'really',
 'does',
 'help',
 'improve',
 'fertility',
 'and',
 'effectiveness',
 'of',
 'sperm',
 'and',
 'semen',
 'motility',
 '.',
 'I',
 'used',
 'it',
 'for',
 'the',
 'past',
 'few',
 'months',
 ',',
 'and',
 'not',
 'only',
 'does',
 'it',
 'work',
 '-',
 'I',
 'al

In [30]:
pprint(word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-21])),
       compact=True)

['``', 'I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for',
 'Spur-M', '.', 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and',
 'motility', '.', 'I', 'found', 'your', 'site', 'and', 'ordered', 'Spur-M',
 'Fertility', 'Blend', 'for', 'Men', '.', 'I', 'have', 'wondered', 'for',
 'years', 'what', 'caused', 'low', 'semen', 'and', 'sperm', 'count', ',', 'and',
 'how', 'I', 'could', 'improve', 'my', 'fertility', 'and', 'help', 'my', 'wife',
 'conceive', '.', 'Spur-M', 'seems', 'to', 'have', 'done', 'just', 'that', '!',
 'Thank', 'you', 'for', 'your', 'support', '.', "''", 'Andrew', 'H.', ',',
 'London', ',', 'UK', "''", 'Spur-M', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 '.', 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', ',', 'and',
 'not', 'only', 'does', 'it', 'work', '-', 'I', 'also', 'feel', 'better', 'to',
 '.', 'I', 'have', 'more', 'energy', '.', 'This', 'is', 'an', 'excellent

La eliminación de los caracteres no alfanuméricos se puede realizar mediante expresiones regulares, usando para ello el paquete [re](https://docs.python.org/es/3/library/re.html) de la biblioteca estándar de Python.

In [31]:
import re

**Nota de clase**:
- La biblioteca deja al programador implementar la función `procesa_mensaje`, por lo que se pueden usar distintas técnicas de preprocesado para mejorar el rendimiento del filtro. En este apartado se pide eliminar los caracteres no alfanuméricos, pero se podrían aplicar otras técnicas como convertir los tókenes a minúsculas, eliminar las palabras vacías (stop words), aplicar lematización o stemming, etc.

In [32]:
def elimina_no_alfanumerico(contenido):
    return [re.sub(r'[^\w]', '', palabra)
            for palabra in contenido
            if re.search(r'\w', palabra)]

In [34]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [35]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),
       compact=True)

['I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for', 'SpurM',
 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and', 'motility', 'I',
 'found', 'your', 'site', 'and', 'ordered', 'SpurM', 'Fertility', 'Blend',
 'for', 'Men', 'I', 'have', 'wondered', 'for', 'years', 'what', 'caused', 'low',
 'semen', 'and', 'sperm', 'count', 'and', 'how', 'I', 'could', 'improve', 'my',
 'fertility', 'and', 'help', 'my', 'wife', 'conceive', 'SpurM', 'seems', 'to',
 'have', 'done', 'just', 'that', 'Thank', 'you', 'for', 'your', 'support',
 'Andrew', 'H', 'London', 'UK', 'SpurM', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', 'and', 'not', 'only',
 'does', 'it', 'work', 'I', 'also', 'feel', 'better', 'to', 'I', 'have', 'more',
 'energy', 'This', 'is', 'an', 'excellent', 'counter', 'to', 'low', 'sperm',
 'count', 'and', 'motility', 'I', 'll', 'be', 'buying', 'm

Debido a la naturaleza de los mensajes no deseados, algunos de ellos pueden confundir a la biblioteca Beautiful Soup, avisando esta de que el mensaje puede tratarse de una URL o de una ruta a un fichero, en lugar de un mensaje de correo electrónico. El código de la siguiente celda filtra ese tipo de avisos.

In [36]:
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

Estamos ya en condiciones de poder construir el filtro de correo electrónico no deseado.

In [37]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer # Convertir a vector tf-idf
from sklearn.neighbors import KNeighborsClassifier

In [39]:

vectorizador = TfidfVectorizer(analyzer=procesa_mensaje)
vectorizador.fit(contenidos_mensajes_entrenamiento)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",<function pro...00200739A0A40>
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer

In [40]:
# total de rasgos
len(vectorizador.get_feature_names_out()) # Lo usamos para saber el número de rasgos que se han extraído de los mensajes de entrenamiento, es decir, el tamaño del vocabulario que se ha construido a partir de los mensajes de entrenamiento.

171221

In [41]:
contenidos_mensajes_entrenamiento[-21]

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [42]:
# obtener la representacion de 1 documento
tfidf = vectorizador.transform([contenidos_mensajes_entrenamiento[-21]])

In [43]:
# valores internos de tfidf
print(tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 150 stored elements and shape (1, 171221)>
  Coords	Values
  (0, 2972)	0.034484616367522
  (0, 27073)	0.054536963455873035
  (0, 28178)	0.0438099182176285
  (0, 30177)	0.07335954600793722
  (0, 30398)	0.07531821941864801
  (0, 31291)	0.042891765069407645
  (0, 38573)	0.06974950599581041
  (0, 41830)	0.08023673678243118
  (0, 43420)	0.07588016321730563
  (0, 44153)	0.07251630731903697
  (0, 45484)	0.058709981837101355
  (0, 46489)	0.04541650652083856
  (0, 48927)	0.2893622235538194
  (0, 50788)	0.027907769116665482
  (0, 51914)	0.04958443051673252
  (0, 55086)	0.043365478637554335
  (0, 56800)	0.05997371621910099
  (0, 57958)	0.06119658347111242
  (0, 59311)	0.09515499321222422
  (0, 59396)	0.037606120179473734
  (0, 68916)	0.09515499321222422
  (0, 69474)	0.06974950599581041
  (0, 74514)	0.3256746258437044
  (0, 75271)	0.07780482105265907
  (0, 77212)	0.03357215230920081
  :	:
  (0, 154590)	0.036658462163257195
  (0, 154639)

In [44]:
# vamos a buscar las palabras que corresponden a cada posición, e imprimir el valor de tfidf para cada una de ellas
palabras = vectorizador.get_feature_names_out()

indices_no_cero = tfidf.nonzero()[1] # mat5riz esparcida, obtenemos los índices de las columnas que tienen valores no cero, es decir, las palabras que aparecen en el documento y su valor de tf-idf correspondiente.

for i in indices_no_cero:
    print(f'{palabras[i]}: {tfidf[0, i]}')

100: 0.034484616367522
Andrew: 0.054536963455873035
B: 0.0438099182176285
Blend: 0.07335954600793722
Bonn: 0.07531821941864801
But: 0.042891765069407645
Doctors: 0.06974950599581041
Essex: 0.08023673678243118
Fertility: 0.07588016321730563
Franz: 0.07251630731903697
Germany: 0.058709981837101355
H: 0.04541650652083856
I: 0.2893622235538194
It: 0.027907769116665482
K: 0.04958443051673252
London: 0.043365478637554335
Male: 0.05997371621910099
Men: 0.06119658347111242
Munozprovencauxnetrmphp: 0.09515499321222422
My: 0.037606120179473734
Richardprovencauxnetspur: 0.09515499321222422
Roy: 0.06974950599581041
SpurM: 0.3256746258437044
Suffice: 0.07780482105265907
Thank: 0.03357215230920081
Thanks: 0.024390535836932927
This: 0.02354710227088937
UK: 0.09669339361566667
Usually: 0.0771178969262048
a: 0.02869448484315645
able: 0.07223766116163874
also: 0.027999935553830478
am: 0.027953737781320796
an: 0.022095471176530563
and: 0.16097302305298294
any: 0.021601494695949842
aware: 0.04357615211253

**Nota de clase**:
- Un itf-idf bajo indica que la palabra es común en el corpus, mientras que un itf-idf alto indica que la palabra es rara en el corpus. Por lo tanto, las palabras con un itf-idf alto pueden ser más útiles para distinguir entre mensajes no deseados y mensajes legítimos, ya que son menos comunes y pueden estar más asociadas a los mensajes no deseados. Por otro lado, las palabras con un itf-idf bajo pueden ser menos útiles para distinguir entre mensajes no deseados y mensajes legítimos, ya que son más comunes y pueden aparecer tanto en mensajes no deseados como en mensajes legítimos.

In [ ]:
# buscar la aparicion de "00" en contenidos_mensajes_entrenamiento[-21]
# buscar la seccion donde aparezca, imprimir 30 caracteres hacia atras y adelante

mensaje = contenidos_mensajes_entrenamiento[-21]
patron = "00"

# Buscar todas las ocurrencias de "00"
indice = 0
ocurrencias = []

while indice < len(mensaje):
    posicion = mensaje.find(patron, indice)
    if posicion == -1:
        break
    ocurrencias.append(posicion)
    indice = posicion + 1

# Imprimir el contexto de cada ocurrencia
print(f"Se encontraron {len(ocurrencias)} ocurrencias de '{patron}':\n")
for i, pos in enumerate(ocurrencias, 1):
    inicio = max(0, pos - 60)
    fin = min(len(mensaje), pos + len(patron) + 60)
    contexto = mensaje[inicio:fin]
    print(f"Ocurrencia {i} (posición {pos}):")
    print(f"...{contexto}...")
    print()

In [ ]:
# mostrar las 10 palabras con mayor valor de tfidf

palabras_con_valores = [(palabras[i], tfidf[0, i]) for i in indices_no_cero]

palabras_ordenadas = sorted(palabras_con_valores, key=lambda x: x[1], reverse=True)

print("Las 10 palabras con mayor valor TF-IDF:\n")
for palabra, valor in palabras_ordenadas[:10]:
    print(f'{palabra}: {valor:.6f}')

**Nota de clase**:
- Si alguna palabra clave es "rara", por ejemplo, 33... puede ser que se haya colado algo en el procesado de los mensajes, por lo que es importante revisar el vocabulario aprendido para detectar posibles errores en el preprocesado de los mensajes.
- se puede considerar como una comprobación rápida del preprocesado de los mensajes el revisar el vocabulario aprendido por el vectorizador tf-idf, para detectar posibles errores en el preprocesado de los mensajes. Por ejemplo, si se detecta que hay palabras clave que son números o secuencias de caracteres sin sentido, puede ser un indicio de que se ha colado algo en el procesado de los mensajes, como por ejemplo, que no se han eliminado correctamente los caracteres no alfanuméricos.

In [ ]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [ ]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

In [ ]:
from sklearn.metrics import recall_score

In [ ]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report)

print("=" * 60)
print("RESUMEN DE EVALUACIÓN DEL FILTRO ANTISPAM")
print("=" * 60)

# Métricas individuales
accuracy = accuracy_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
precision = precision_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
recall = recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)
f1 = f1_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

print(f"\nMÉTRICAS DE CLASIFICACIÓN:")
print(f"  Exactitud (Accuracy):   {accuracy:.4f}")
print(f"  Precisión (Precision):  {precision:.4f}")
print(f"  Sensibilidad (Recall):  {recall:.4f}")
print(f"  F1-Score:               {f1:.4f}")

# Matriz de confusión
print(f"\nMATRIZ DE CONFUSIÓN:")
cm = confusion_matrix(clases_mensajes_prueba, predicciones_mensajes_prueba)
print(f"                    Predicho: Legítimo  Predicho: Spam")
print(f"  Real: Legítimo           {cm[0][0]:6d}          {cm[0][1]:6d}")
print(f"  Real: Spam               {cm[1][0]:6d}          {cm[1][1]:6d}")

# Reporte de clasificación completo
print(f"\nREPORTE DETALLADO:")
print(classification_report(clases_mensajes_prueba, predicciones_mensajes_prueba, 
                           target_names=['Legítimo', 'No deseado']))

print("=" * 60)

#### Apartado 1

En este apartado se pide incorporar al procesado de mensajes los siguientes 2 pasos:

* Expandir las contracciones típicas del idioma inglés. Usar para ello el paquete [contractions](https://github.com/kootenpv/contractions).
* Convertir todos los caracteres a minúsculas.

In [ ]:

import contractions

#contractions.fix("I'm")

def expande_contraccion(contenido):
    return [contractions.fix(palabra) for palabra in contenido]
    

def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contraccion(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-21]),compact=True)

TypeError: expected string or bytes-like object, got 'list'

#### Apartado 2

Palabras vacías (_stop words_, en inglés) es el nombre que reciben las palabras tales como artículos, pronombres y preposiciones que se considera que no aportan significado para un sistema de procesamiento del lenguaje natural y que, por tanto, deben eliminarse durante las operaciones de preprocesado de texto. El conjunto adecuado de palabras vacías a usar depende del sistema concreto que se esté construyendo, e incluso puede resultar conveniente no hacer uso de esta técnica.

NLTK provee de conjuntos genéricos de palabras vacías para distintos idiomas.

In [ ]:
download('stopwords', download_dir='.')

In [ ]:
from nltk.corpus import stopwords
from nltk.data import path
path.append(".")

In [ ]:
palabras_vacias_ingles = stopwords.words('english')
pprint(palabras_vacias_ingles, compact=True)

En este apartado se pide incorporar al procesado de mensajes la eliminación de palabras vacías.

In [ ]:
def elimina_palabras_vacias(contenido, palabras_vacias_ingles)
    return []

def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = expande_contraccion(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

procesa_mensaje(contenidos_mensajes_entrenamiento)

#### Apartado 3

Por razones gramaticales, en un documento de texto van a aparecer con seguridad diferentes formas de una palabra, como organizar, organiza y organizando. Además, existen familias de palabras relacionadas derivativamente con significados similares, como democracia, democrático y democratización. En muchas situaciones, parece que sería útil reducir esos conjuntos de palabras a una raíz común. Para ello se suelen usar los procedimientos de _stemming_ y lematización.

_Stemming_ generalmente se refiere a un proceso heurístico rudimentario que corta los extremos de las palabras con la esperanza de lograr el objetivo correctamente la mayor parte del tiempo y, a menudo, incluye la eliminación de afijos derivativos. La lematización generalmente se refiere a hacer las cosas correctamente con el uso de un vocabulario y análisis morfológico de las palabras, normalmente con el objetivo de eliminar únicamente las terminaciones flexivas y devolver la forma base o de diccionario de una palabra, lo que se conoce como lema.

NLTK provee de varios algoritmos de _stemming_ y lematización. En este apartado se pide incorporar al procesado de mensajes el procedimiento de _stemming_ mediante el [algoritmo de Lancaster](https://www.nltk.org/api/nltk.stem.lancaster.html).

### Ejercicio 2

En el cuaderno NLTK.ipynb se ha construido un sistema de predicción de texto en español basado en modelos de $n$-gramas. Estos modelos se han entrenado a partir de un corpus de textos en español que se ha usado en bruto. El objetivo de este ejercicio es recrear la construcción del sistema de predicción de texto, pero usando una versión normalizada del corpus.

#### Apartado 1

En este apartado se pide:

1. Leer el corpus guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt` y dividirlo en un corpus de entrenamiento y un corpus de prueba.
2. Construir modelos unigramas, bigramas y trigramas, con y sin suavizado, a partir del corpus de entrenamiento normalizado convirtiendo todas las palabras a minúsculas.
3. Seleccionar el modelo con menor perplejidad sobre el corpus de prueba normalizado convirtiendo todas las palabras a minúsculas.

In [ ]:
# Nos aseguramos de haber descargado el tokenizador

from nltk import download

download('punkt', download_dir='.')

In [ ]:
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.data import load

In [ ]:
corpus_InfoLibros = PlaintextCorpusReader(
    root='Texto predictivo',
    fileids=['corpus_InfoLibros_parcial.txt'],
    encoding='utf8',
    sent_tokenizer=load('tokenizers/punkt/spanish.pickle')
)

In [ ]:
total_frases = len(corpus_InfoLibros.sents())
total_frases

In [ ]:
total_frases_entrenamiento = int(total_frases * .8)
total_frases_entrenamiento

In [ ]:
corpus_entrenamiento = corpus_InfoLibros.sents()[:total_frases_entrenamiento]

In [ ]:
corpus_prueba = corpus_InfoLibros.sents()[total_frases_entrenamiento:]

In [ ]:
from nltk.lm.vocabulary import Vocabulary
from nltk.lm.preprocessing import flatten

In [ ]:
vocabulario_palabras = Vocabulary(
    (palabra.lower()
     for palabra in flatten(corpus_entrenamiento)),  # lista de todas las palabras
    unk_cutoff=50  # mínimo número de ocurrencias
)

In [ ]:
vocabulario_palabras.lookup('hola')

In [ ]:
inicio_frase = '<s>'
fin_frase = '</s>'
vocabulario_palabras.update({inicio_frase: 50, fin_frase: 50})

In [ ]:
def delimita_frase(frase, n):
    return (['<s>'] * (n - 1) +
            [palabra.lower() for palabra in frase] +
            ['</s>'])

In [ ]:
from pprint import pprint

In [ ]:
primera_frase_entrenamiento = corpus_entrenamiento[0]
pprint(delimita_frase(primera_frase_entrenamiento, 1),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 2),
       compact=True)
pprint(delimita_frase(primera_frase_entrenamiento, 3),
       compact=True)

In [ ]:
from nltk.util import ngrams, bigrams, trigrams
from nltk.lm import MLE, Laplace

In [ ]:
for ii in ngrams(delimita_frase(corpus_entrenamiento[0], 1), n=1):
    print(ii)

In [ ]:
modelo_unigrama_MLE = MLE(1, vocabulary=vocabulario_palabras)
modelo_unigrama_MLE.fit
modelo_unigrama_MLE.perplexity(flatten())

In [ ]:
modelo_unigrama_Laplace = Laplace(1, vocabulary=vocabulario_palabras)
modelo_unigrama_Laplace.fit
modelo_unigrama_Laplace.perplexity(flatten())

In [ ]:
modelo_bigrama_MLE = MLE(2, vocabulary=vocabulario_palabras)
modelo_bigrama_MLE.fit
modelo_bigrama_MLE.perplexity(flatten())

In [ ]:
modelo_bigrama_Laplace = Laplace(2, vocabulary=vocabulario_palabras)
modelo_bigrama_Laplace.fit
modelo_bigrama_Laplace.perplexity(flatten())

In [ ]:
modelo_trigrama_MLE = MLE(3, vocabulary=vocabulario_palabras)
modelo_trigrama_MLE.fit
modelo_trigrama_MLE.perplexity(flatten())

In [ ]:
modelo_trigrama_Laplace = Laplace(3, vocabulary=vocabulario_palabras)
modelo_trigrama_Laplace.fit
modelo_trigrama_Laplace.perplexity(flatten())

#### Apartado 2

Definir una función `predice_palabras` que prediga, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir. La función debe actuar como sigue:

* Si todas las letras del prefijo escrito están en minúsculas, entonces debe predecir palabras en minúsculas.
* Si todas las letras del prefijo escrito están en mayúsculas, entonces debe predecir palabras en mayúsculas.
* Si el prefijo escrito mezcla letras en minúsculas y en mayúsculas, entonces:
  * Si la primera letra del prefijo está en minúsculas, entonces debe predecir palabras en minúsculas.
  * Si la primera letra del prefijo está en mayúsculas, entonces debe predecir palabras con la primera letra en mayúsculas y el resto en minúsculas.

In [ ]:
predice_palabras('nat', ('Lenguaje',), 5, modelo_bigrama_Laplace)